In [19]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1. Load Dataset dan EDA Singkat

Pada tahap ini dataset Titanic dimuat menggunakan library Seaborn. Kemudian dilakukan pengecekan ukuran data, tipe data, missing values, dan distribusi target.

In [20]:
df = sns.load_dataset('titanic')

cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']
df = df[cols].copy()

print('Shape:', df.shape)

print('\n5 Data Teratas:')
display(df.head())

print('\nInfo Dataset:')
df.info()

Shape: (891, 8)

5 Data Teratas:


,pclass,sex,age,sibsp,parch,fare,embarked,survived
0,3,male,22.0,1,0,7.2500,S,0
1,1,female,38.0,1,0,71.2833,C,1
2,3,female,26.0,0,0,7.9250,S,1
3,1,female,35.0,1,0,53.1000,S,1
4,3,male,35.0,0,0,8.0500,S,0



Info Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    891 non-null    int64  
 1   sex       891 non-null    object 
 2   age       714 non-null    float64
 3   sibsp     891 non-null    int64  
 4   parch     891 non-null    int64  
 5   fare      891 non-null    float64
 6   embarked  889 non-null    object 
 7   survived  891 non-null    int64  
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [21]:
print('Missing values:')
print(df.isnull().sum())

print('\nDistribusi target survived:')
print(df['survived'].value_counts())

print('\nDistribusi target dalam persen:')
print(df['survived'].value_counts(normalize=True).round(3))

Missing values:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

Distribusi target survived:
survived
0    549
1    342
Name: count, dtype: int64

Distribusi target dalam persen:
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


## 2. Handling Missing Values

Missing value pada kolom numerik `age` diisi menggunakan median, sedangkan missing value pada kolom kategorikal `embarked` diisi menggunakan modus.

In [22]:
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print('Missing values setelah handling:')
print(df.isnull().sum())

Missing values setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


## 3. Encoding Data Kategorikal

Kolom kategorikal `sex` dan `embarked` diubah menjadi bentuk numerik menggunakan One-Hot Encoding dengan `drop_first=True`.

In [23]:
df = pd.get_dummies(
    df,
    columns=['sex', 'embarked'],
    drop_first=True,
    dtype=int
)

print('Kolom setelah encoding:')
print(df.columns.tolist())

display(df.head())

Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


,pclass,age,sibsp,parch,fare,survived,sex_male,embarked_Q,embarked_S
0,3,22.0,1,0,7.2500,0,1,0,1
1,1,38.0,1,0,71.2833,1,0,0,0
2,3,26.0,0,0,7.9250,1,0,0,1
3,1,35.0,1,0,53.1000,1,0,0,1
4,3,35.0,0,0,8.0500,0,1,0,1


## 4. Train-Test Split

Dataset dibagi menjadi data latih dan data uji dengan proporsi 80:20. Parameter `stratify=y` digunakan agar proporsi kelas target tetap seimbang pada data train dan test.

In [24]:
X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape[0]} baris')
print(f'Test : {X_test.shape[0]} baris')

print('\nProporsi survived di Train:')
print(y_train.value_counts(normalize=True).round(3))

print('\nProporsi survived di Test:')
print(y_test.value_counts(normalize=True).round(3))

Train: 712 baris
Test : 179 baris

Proporsi survived di Train:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi survived di Test:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64


## 5. Feature Scaling

Feature scaling dilakukan menggunakan StandardScaler. Scaling hanya diterapkan pada kolom numerik, sedangkan kolom biner hasil encoding tidak perlu di-scale.

In [25]:
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Mean scaler dari data train:')
print(scaler.mean_.round(2))

print('\nStd scaler dari data train:')
print(scaler.scale_.round(2))

print('\nContoh X_train setelah scaling:')
display(X_train.head().round(3))

Mean scaler dari data train:
[ 2.31 29.46  0.49  0.39 31.82]

Std scaler dari data train:
[ 0.83 13.03  1.06  0.84 48.03]

Contoh X_train setelah scaling:


,pclass,age,sibsp,parch,fare,sex_male,embarked_Q,embarked_S
692,0.830,-0.112,-0.465,-0.466,0.514,1,0,1
481,-0.371,-0.112,-0.465,-0.466,-0.663,1,0,1
527,-1.571,-0.112,-0.465,-0.466,3.955,1,0,1
855,0.830,-0.880,-0.465,0.728,-0.468,0,0,1
801,-0.371,0.118,0.478,0.728,-0.116,0,0,1


## 6. Kesimpulan

Pada hands-on ini, dataset Titanic telah melalui proses persiapan data, yaitu:

1. Load dataset dan EDA singkat.
2. Handling missing values pada kolom `age` dan `embarked`.
3. Encoding data kategorikal menggunakan One-Hot Encoding.
4. Train-test split dengan stratifikasi.
5. Feature scaling menggunakan StandardScaler.

Data akhir sudah siap digunakan untuk proses pelatihan model Machine Learning.

In [26]:
print('Data siap digunakan untuk Machine Learning:')
print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'y_test : {y_test.shape}')

Data siap digunakan untuk Machine Learning:
X_train: (712, 8)
y_train: (712,)
X_test : (179, 8)
y_test : (179,)
